# Optuna Optimized LightGBM Model Training

LightGBM converges fast on tabular data and give much better CV score. It uses **histogram-based binning**.

## Import Libraries

Importing all the required libraries at the beginning in advance.

In [ ]:
# Import required libraries

import json
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import TPESampler

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score

from lightgbm import LGBMClassifier

## Load Preprocessed Data

Loading preprocessed data

In [ ]:
# Reading the preprocessed dataset

processed_train = pd.read_parquet(
    "/kaggle/input/datasets/shivamgravity/pgs-s6e6-processed-data-v1/processed_train.parquet"
)
processed_test = pd.read_parquet(
    "/kaggle/input/datasets/shivamgravity/pgs-s6e6-processed-data-v1/processed_test.parquet"
)

## Configs

Explictly writing settings and parameters for further use.

In [ ]:
# Configs

# Target feature
TARGET = "class"
ID = "id"

# Test ids - used to create submission files after prediction
TEST_ID = processed_test[ID]

# Categorical columns
cat_cols = [
    "spectral_type",
    "galaxy_population"
]

# Optuna and CV configs
N_SPLITS = 7
RANDOM_STATE = 42
N_TRIALS = 50

## Data Preparation For Training & Testing

Splitting target feature from train dataset early, to manage the training further.

Removing the ID feature from train and test dataset both.

In [ ]:
# Preparing the datasets for training and testing purpose

# Removing the id and target feature
X = processed_train.drop([ID,TARGET], axis=1).copy()
y = processed_train[TARGET]

# Removing the id feature
X_test = processed_test.drop([ID], axis=1).copy()

## Encoding Categorical Feature

Encoding **TARGET** and **other categorical features**.

It allows LightGBM to access the values in numeric format.

In [ ]:
# Encoding

# Encoding target feature
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y)

# Encoding categorical features beside target feature
feature_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

    feature_encoders[col] = le

## Training Sample

Extracting the training sample before using Optuna for hyperparameter optimization.

Optuna will run many trials and each trial will have multiple folds, a miniature LGBM model will be trained each will.

It will take a lot of time and resource to train and tune the model and hyperparameters, repectively.

Therefore, using 100k rows to get tune the hyperparameters is a better option.

It will give 90 to 95% similar hyperparameters in a fractiion of cost.

In [ ]:
# Getting the train data sample for optimization

X_sample, _, y_sample, _ = train_test_split(
    X,
    y,
    train_size=100_000,
    random_state=RANDOM_STATE,
    stratify=y
)

## Optuna Optimization with CV

Optuna will search for optimized params very fast.

To improve CV Score, I am using **StratifiedKFold**.

It makes sure that every fold has **equal distribution** of **classes**.

In [ ]:
# =====================================================
# Optuna Objective
# =====================================================

def objective(trial):

    params = {

        "objective": "multiclass",
        "num_class": len(np.unique(y)),

        "n_estimators": trial.suggest_int(
            "n_estimators",
            300,
            2000
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.2,
            log=True
        ),

        "num_leaves": trial.suggest_int(
            "num_leaves",
            20,
            255
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            15
        ),

        "min_child_samples": trial.suggest_int(
            "min_child_samples",
            5,
            100
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.5,
            1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.5,
            1.0
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-8,
            10.0,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1e-8,
            10.0,
            log=True
        ),

        "random_state": 42,
        "verbosity": -1
    }

    fold_scores = []

    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    for train_idx, valid_idx in skf.split(X_sample, y_sample):

        X_train = X_sample.iloc[train_idx]
        X_valid = X_sample.iloc[valid_idx]

        y_train = y_sample[train_idx]
        y_valid = y_sample[valid_idx]

        model = LGBMClassifier(**params)

        model.fit(
            X_train,
            y_train
        )

        preds = model.predict(X_valid)

        score = balanced_accuracy_score(
            y_valid,
            preds
        )

        fold_scores.append(score)

    return np.mean(fold_scores)

In [ ]:
# =====================================================
# Optuna Optimization
# =====================================================

study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=42)
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True
)

best_params = study.best_params

print("\nBest Score:")
print(study.best_value)

print("\nBest Params:")
print(best_params)

In [ ]:
# Save Best Params

with open("best_params.json", "w") as f:
    json.dump(best_params, f, indent=4)

In [ ]:
# Save the whole Optuna study results

optuna_results = study.trials_dataframe()

optuna_results.to_csv(
    "optuna_results.csv",
    index=False
)

## Final Model Training

Training the final_model to predict on test dataset.

In [ ]:
final_params = best_params.copy()

final_params.update({
    "objective": "multiclass",
    "num_class": len(np.unique(y)),
    "random_state": 42,
    "verbosity": -1
})

final_model = LGBMClassifier(
    **final_params
)

final_model.fit(
    X,
    y
)

## Prediction On Test Data

The final trained LightGBM model with best params will be used to predict the test data.

In [ ]:
# Prediction on test data

# Predicting the values
pred = final_model.predict(X_test)

# Getting the associated labels with the numeric value predictions
pred_labels = target_encoder.inverse_transform(pred.astype(int))

## Competition Submission File

Saving the prediction as csv file to submit in the competition.

In [ ]:
# Saving the results

# Creating result dataframe
submission = pd.DataFrame({
    "id": TEST_ID,
    "class": pred_labels
})

# Saving the submission file
submission.to_csv("submission.csv", index=False)